In [39]:
from langchain.document_loaders import YoutubeLoader
# ! pip install youtube-transcript-api
# ! pip install pytube
# "https://www.youtube.com/watch?v=lcyHC9HLTzc",
urls = ["https://www.youtube.com/watch?v=K9mzg8ueiYA"]
save_dir = "docs/youtube/"
docs = []
for url in urls:
    loader = YoutubeLoader.from_youtube_url(url,add_video_info = True)
    docs += (loader.load())
    

In [ ]:
docs

In [59]:
len(docs[0].page_content.split())

557

In [58]:
xx = "hi there my name is usman"
x = xx.split()
print(len(x))

6


In [72]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

r_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100, 
    length_function=len,
    )

docsvec = r_splitter.split_documents(docs)

In [ ]:
docsvec

In [75]:
len(docsvec)

4

In [78]:
from dotenv import load_dotenv
load_dotenv()

True

In [79]:
import os
from dotenv import load_dotenv
load_dotenv()
import pickle
import faiss
from langchain.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings
embeddings = OpenAIEmbeddings()
vectorstore_openAI = FAISS.from_documents(docsvec, embeddings)
with open("faiss_store_openai.pkl", "wb") as f:
    pickle.dump(vectorstore_openAI, f)

In [ ]:
from langchain.chat_models import ChatOpenAI
llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)
llm

In [82]:
import pickle
with open("faiss_store_openai.pkl", "rb") as f:
    vectorstore = pickle.load(f)

In [ ]:
question = "who created pascal"
docsvec = vectorstore.similarity_search(question,k = 2)
print(docsvec)

In [105]:
from langchain.memory import ConversationSummaryBufferMemory
memory = ConversationSummaryBufferMemory(llm=llm, max_token_limit=100,memory_key="chat_history",return_messages=True)

In [ ]:
memory

In [118]:
from langchain.prompts import PromptTemplate
template = """Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you its not  in the video, don't try to make up an answer.Keep the answer as concise as possible. Always say "thanks for asking!" on the next line at the end of the answer. 
{context}
Question: {question}
Helpful Answer:"""
QA_CHAIN_PROMPT = PromptTemplate(input_variables=["context", "question"], template=template)

In [136]:
qa = "hi"

In [147]:
from langchain.chains import RetrievalQA
from langchain.chains import ConversationalRetrievalChain
retriever = vectorstore.as_retriever()
'''
qa = ConversationalRetrievalChain.from_chain_type(
    llm,
    retriever=retriever,
    memory=memory,
    return_source_documents=True,
    chain_type_kwargs={"prompt": QA_CHAIN_PROMPT}
)
'''

qa_chain = RetrievalQA.from_chain_type(llm,
                                       retriever=retriever,
                                       #return_source_documents=True,
                                       chain_type_kwargs={"prompt": QA_CHAIN_PROMPT},
                                       memory = memory)

In [ ]:
qa_chain

In [139]:
qa

'hi'

In [ ]:
question = "which programming language is pascal like "
result = qa_chain({"query": question})

print(result)

In [ ]:
print(result)

In [ ]:
memory.load_memory_variables({})

In [ ]:
result["result"]